# Demo Incremental DBSCAN

Notebook này minh họa phần cốt lõi của đồ án theo thứ tự:

1. Insert làm hai cụm gộp lại.
2. Delete điểm cầu nối làm một cụm tách ra.
3. DBSCAN trên bản đồ ATM 2D.
4. Đánh giá bằng Silhouette tự cài và so sánh với các thuật toán mật độ của scikit-learn.

Dữ liệu đều là mô phỏng. Phần so sánh 2D/3D/4D và thời gian cập nhật nằm riêng trong So_sanh_thuat_toan.ipynb.

## 0. Chuẩn bị môi trường

In [1]:
import os
import platform
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "clustering").is_dir():
            return candidate
    raise FileNotFoundError("Mở notebook từ thư mục repo hoặc thư mục con của repo.")


ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", None)
print("Repo:", ROOT.name, "| Python:", platform.python_version(), "| NumPy:", np.__version__, "| Pandas:", pd.__version__)


Repo: INCREMENTAL_DBSCAN_NHOM_1 | Python: 3.14.2 | NumPy: 2.4.3 | Pandas: 3.0.3


In [2]:
from sklearn.cluster import DBSCAN as SklearnDBSCAN
from sklearn.cluster import HDBSCAN, OPTICS
from sklearn.metrics import adjusted_rand_score

from src.clustering.dbscan import DBSCAN
from src.evaluation.metrics import silhouette_score as custom_silhouette_score
from src.experiments.engine import IncrementalEngine
from src.experiments.update_cases import update_seeds
from src.visualization import ClusterVisualizer, load_scenario


def silhouette_or_nan(coords, labels):
    """Silhouette tự cài; noise bị bỏ qua, NaN nếu không đủ hai cụm."""
    labels = np.asarray(labels)
    n_clusters = len(set(labels.tolist()) - {-1})
    n_scored = int(np.count_nonzero(labels != -1))
    if n_clusters < 2 or n_scored <= n_clusters:
        return np.nan
    return custom_silhouette_score(coords, labels)


## 1. Insert: điểm mới làm hai cụm nhập thành một

Ví dụ R02 dùng 12 điểm 2D, Eps=1.1, MinPts=4. Điểm p được chèn vào giữa hai cụm.

In [3]:
sc_merge = load_scenario("toy", "r02_4_merge")
pd.DataFrame(sc_merge["coords"], index=sc_merge["names"], columns=["x", "y"]).T

,A1,A2,A3,A4,A5,A6,B1,B2,B3,B4,B5,B6
x,-1.0,0.0,1.0,-1.0,0.0,1.0,2.8,3.8,4.8,2.8,3.8,4.8
y,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0


In [4]:
toy = IncrementalEngine(sc_merge["eps"], sc_merge["min_pts"]).fit(sc_merge["coords"])
before_merge = toy.snapshot()

pd.DataFrame({
    "Điểm": sc_merge["names"],
    "Số láng giềng": [toy.points[i].neighbor_count for i in range(len(sc_merge["names"]))],
    "Loại điểm": [toy.points[i].point_type.value for i in range(len(sc_merge["names"]))],
    "Cụm trước insert": before_merge["labels"],
})

,Điểm,Số láng giềng,Loại điểm,Cụm trước insert
0,A1,3,BORDER,0
1,A2,4,CORE,0
2,A3,3,BORDER,0
3,A4,3,BORDER,0
4,A5,4,CORE,0
5,A6,3,BORDER,0
6,B1,3,BORDER,1
7,B2,4,CORE,1
8,B3,3,BORDER,1
9,B4,3,BORDER,1


In [5]:
new_id = toy.insert(sc_merge["updates"][0]["coords"])
after_merge = toy.snapshot()
names_after_insert = sc_merge["names"] + ["p"]
seed_ids, newly_core = update_seeds(before_merge, after_merge, "insert", sc_merge["eps"])

print("ID điểm mới:", new_id)
print("Điểm vừa thành core:", sorted(names_after_insert[i] for i in newly_core))
print("UpdSeed:", sorted(names_after_insert[i] for i in seed_ids))
print("Đáp án kịch bản:", sorted(sc_merge["expected"]["updseed"]))

pd.DataFrame({
    "Điểm": names_after_insert,
    "Cụm sau insert": after_merge["labels"],
    "Loại điểm sau insert": [toy.points[i].point_type.value for i in range(len(names_after_insert))],
})

ID điểm mới: 12
Điểm vừa thành core: ['A3', 'A6', 'B1', 'B4', 'p']
UpdSeed: ['A2', 'A3', 'A5', 'A6', 'B1', 'B2', 'B4', 'B5', 'p']
Đáp án kịch bản: ['A2', 'A3', 'A5', 'A6', 'B1', 'B2', 'B4', 'B5', 'p']


,Điểm,Cụm sau insert,Loại điểm sau insert
0,A1,0,BORDER
1,A2,0,CORE
2,A3,0,CORE
3,A4,0,BORDER
4,A5,0,CORE
5,A6,0,CORE
6,B1,0,CORE
7,B2,0,CORE
8,B3,0,BORDER
9,B4,0,CORE


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
ClusterVisualizer(point_size=34).plot_before_after(
    before_merge,
    after_merge,
    {"op": "insert", "point_id": new_id, "coords": sc_merge["updates"][0]["coords"]},
    eps=sc_merge["eps"],
    seed_ids=seed_ids,
    axes=axes,
    suptitle="Insert điểm p làm hai cụm nhập thành một",
)
plt.show()


C:\INCREMENTAL_DBSCAN_NHOM_1\tmp\notebook_check\tmp\ipykernel_23036\3250940779.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Delete: xóa điểm cầu nối làm cụm tách đôi

Ví dụ R03 Case 3A dùng điểm p làm cầu nối. Sau khi xóa p, hai phía không còn nối qua chuỗi điểm core.

In [7]:
sc_split = load_scenario("toy", "r03_3a_split")
toy_split = IncrementalEngine(sc_split["eps"], sc_split["min_pts"]).fit(sc_split["coords"])
before_split = toy_split.snapshot()
names_split = sc_split["names"]
deleted_id = names_split.index("p")

toy_split.delete(deleted_id)
after_split = toy_split.snapshot()
labels_after_by_id = dict(zip(after_split["ids"].tolist(), after_split["labels"].tolist()))
seed_ids, lost_core = update_seeds(before_split, after_split, "delete", sc_split["eps"])

print("Xóa điểm:", names_split[deleted_id])
print("Điểm mất trạng thái core:", sorted(names_split[i] for i in lost_core))
print("UpdSeed:", sorted(names_split[i] for i in seed_ids))
print("Đáp án kịch bản:", sorted(sc_split["expected"]["updseed"]))

pd.DataFrame({
    "Điểm": [name for i, name in enumerate(names_split) if i != deleted_id],
    "Cụm trước delete": [before_split["labels"][i] for i in range(len(names_split)) if i != deleted_id],
    "Cụm sau delete": [labels_after_by_id[i] for i in range(len(names_split)) if i != deleted_id],
}).set_index("Điểm").T

Xóa điểm: p
Điểm mất trạng thái core: ['p']
UpdSeed: ['A', 'E']
Đáp án kịch bản: ['A', 'E']


Điểm,A,B,C,D,E,F,G,H,I
Cụm trước delete,0,0,0,0,0,0,0,0,0
Cụm sau delete,0,0,0,0,1,1,1,1,-1


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
ClusterVisualizer(point_size=34).plot_before_after(
    before_split,
    after_split,
    {"op": "delete", "point_id": deleted_id, "coords": before_split["coords"][deleted_id]},
    eps=sc_split["eps"],
    seed_ids=seed_ids,
    axes=axes,
    suptitle="Delete điểm p làm cụm tách thành hai",
)
plt.show()


C:\INCREMENTAL_DBSCAN_NHOM_1\tmp\notebook_check\tmp\ipykernel_23036\2168459989.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. DBSCAN trên dữ liệu ATM 2D

Bộ ATM gồm 1.200 giao dịch mô phỏng. Dù CSV có nhiều cột mô tả, thuật toán chỉ nhận hai tọa độ x_km và y_km. Demo dùng Eps=0.5 km để xét giao dịch trong bán kính 500 m và MinPts=4 để yêu cầu ít nhất bốn điểm trong vùng lân cận, tính cả điểm trung tâm. Đây là cấu hình minh họa trên dữ liệu ATM, không phải tham số tối ưu hay cấu hình benchmark.

In [9]:
atm = pd.read_csv("data/raw/atm_txn_2d.csv", encoding="utf-8")
X_atm = atm[["x_km", "y_km"]].to_numpy(dtype=float)

eps_atm = 0.5
min_pts_atm = 4
model_atm = IncrementalEngine(eps_atm, min_pts_atm).fit(X_atm)
labels_oop = np.asarray(model_atm.snapshot()["labels"])

print("Số điểm:", len(X_atm))
print("Số cụm:", len(set(labels_oop.tolist()) - {-1}))
print("Số noise:", int(np.count_nonzero(labels_oop == -1)))
print("Silhouette tự cài (bỏ noise):", round(silhouette_or_nan(X_atm, labels_oop), 4))

Số điểm: 1200
Số cụm: 6
Số noise: 142
Silhouette tự cài (bỏ noise): 0.3362


In [10]:
fig, ax = plt.subplots(figsize=(7, 5.5))
cluster_labels = sorted(set(labels_oop.tolist()) - {-1})
for color_index, cluster_id in enumerate(cluster_labels):
    mask = labels_oop == cluster_id
    ax.scatter(X_atm[mask, 0], X_atm[mask, 1], s=12, alpha=0.8,
               color=plt.get_cmap("tab10")(color_index % 10), label=f"Cụm {cluster_id}")
noise = labels_oop == -1
ax.scatter(X_atm[noise, 0], X_atm[noise, 1], s=18, marker="x", color="black", label="Noise")
ax.set(xlabel="x (km)", ylabel="y (km)", title="Incremental DBSCAN OOP trên ATM 2D")
ax.grid(alpha=0.2)
ax.legend(frameon=False, ncol=2)
plt.show()

C:\INCREMENTAL_DBSCAN_NHOM_1\tmp\notebook_check\tmp\ipykernel_23036\442881537.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Kiểm tra tương đương với DBSCAN của scikit-learn

ARI ở đây chỉ kiểm tra hai cách cài DBSCAN có cho cùng phân hoạch hay không. Đây không phải điểm chất lượng cụm và không dùng nhãn mô phỏng làm đầu vào.

In [11]:
labels_library_dbscan = SklearnDBSCAN(
    eps=eps_atm, min_samples=min_pts_atm, metric="euclidean", n_jobs=1
).fit_predict(X_atm)

print("ARI OOP so với sklearn DBSCAN:", round(adjusted_rand_score(labels_oop, labels_library_dbscan), 4))
print("Tập noise giống nhau:", set(np.flatnonzero(labels_oop == -1)) == set(np.flatnonzero(labels_library_dbscan == -1)))

ARI OOP so với sklearn DBSCAN: 1.0
Tập noise giống nhau: True


## 5. So sánh chất lượng trên cùng bản đồ ATM 2D

Silhouette tự cài nằm trong khoảng [-1, 1]; điểm noise không được tính. Đọc điểm cùng số cụm và tỉ lệ noise: Silhouette cao hơn không tự động có nghĩa là kết quả phù hợp hơn nếu thuật toán chia quá nhiều cụm.

In [12]:
labels_by_algorithm = {
    "Incremental DBSCAN OOP": labels_oop,
    "sklearn DBSCAN": labels_library_dbscan,
    "sklearn HDBSCAN": HDBSCAN(
        min_cluster_size=min_pts_atm,
        min_samples=min_pts_atm,
        metric="euclidean",
        cluster_selection_method="eom",
        n_jobs=1,
    ).fit_predict(X_atm),
    "sklearn OPTICS": OPTICS(
        min_samples=min_pts_atm,
        max_eps=eps_atm + 1e-6,
        metric="euclidean",
        cluster_method="dbscan",
        eps=eps_atm,
        n_jobs=1,
    ).fit_predict(X_atm),
}

comparison_rows = []
for name, labels in labels_by_algorithm.items():
    labels = np.asarray(labels)
    comparison_rows.append({
        "Thuật toán": name,
        "Silhouette tự cài (bỏ noise)": silhouette_or_nan(X_atm, labels),
        "Số cụm": len(set(labels.tolist()) - {-1}),
        "Tỉ lệ noise": float(np.mean(labels == -1)),
    })

comparison_atm = pd.DataFrame(comparison_rows).set_index("Thuật toán")
comparison_atm.round(4)

C:\Users\nguye\AppData\Roaming\Python\Python314\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,Silhouette tự cài (bỏ noise),Số cụm,Tỉ lệ noise
Thuật toán,,,
Incremental DBSCAN OOP,0.3362,6,0.1183
sklearn DBSCAN,0.3362,6,0.1183
sklearn HDBSCAN,0.3978,23,0.0992
sklearn OPTICS,0.3362,6,0.1183


In [13]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
for ax, (name, labels) in zip(axes.flat, labels_by_algorithm.items()):
    labels = np.asarray(labels)
    for color_index, cluster_id in enumerate(sorted(set(labels.tolist()) - {-1})):
        mask = labels == cluster_id
        ax.scatter(X_atm[mask, 0], X_atm[mask, 1], s=10, alpha=0.75,
                   color=plt.get_cmap("tab20")(color_index % 20))
    noise = labels == -1
    if noise.any():
        ax.scatter(X_atm[noise, 0], X_atm[noise, 1], s=14, marker="x", color="black")
    row = comparison_atm.loc[name]
    ax.set_title(
        f"{name}\nSilhouette={row['Silhouette tự cài (bỏ noise)']:.3f}; "
        f"cụm={int(row['Số cụm'])}; noise={row['Tỉ lệ noise']:.1%}"
    )
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.grid(alpha=0.15)
fig.suptitle("Bốn thuật toán trên cùng dữ liệu ATM", y=1.01)
fig.tight_layout()
plt.show()

C:\INCREMENTAL_DBSCAN_NHOM_1\tmp\notebook_check\tmp\ipykernel_23036\1437622871.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Kết luận ngắn

- Nhóm tự cài DBSCAN và cập nhật cục bộ bằng insert/delete theo OOP.
- Hai ví dụ toy trình bày từng bước Merge và Split; ATM minh họa dữ liệu 2D.
- Silhouette tự cài là chỉ số chất lượng chính; DBSCAN, HDBSCAN và OPTICS của scikit-learn là các phương pháp đối chiếu.
- Notebook so sánh dùng một luồng thẻ cho cả 2D, 3D và 4D, không cần các bộ dữ liệu mở rộng riêng.
